# MGT5493 Lab 3 Homework — Exploratory Data Analysis & Visualization

**100 marks** — 85 visible automatic marks and 15 marks for written responses.
Run the setup cell at the start of a fresh kernel, then work through the answer cells and their checks.
Leave the grading cells in place. The last cell summarises the automatic checks.

Use dataset of `marketing_performance.csv`.

## Part 0. Import packages

Running the below homework marking system package installation and import to test your answers

In [ ]:
# %pip install "homework-marking-system[tables,plots] @ git+https://github.com/KrisrcQi/Homework_marking_system.git@v0.2.0"

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from lab3_marking import create_marking

marking = create_marking("marketing_performance.csv")

## Part 1. Load and inspect the dataset (5 marks)

Load the CSV as `df`. Before cleaning, save and show the following outputs (1 mark each).
- `first_5` — the first five rows, copied so later changes to `df` do not affect them.
- `data_shape` — the shape of data 
- `data_statis` — the output of `df.describe()`.
- `data_missing_summary` — missing-value counts for every column.

Fill missing `Region` labels in `df` with `"Unknown"` and keep the other values and rows as supplied (1 mark).

In [ ]:
### BEGIN SOLUTION
df = pd.read_csv("marketing_performance.csv")
first_5 = df.head().copy()
data_shape = df.shape
data_statis = df.describe()
data_missing_summary = df.isna().sum()
display(first_5)
display(data_statis, data_missing_summary)
df["Region"] = df["Region"].fillna("Unknown")
### END SOLUTION

In [ ]:
marking.check('p1-data', df=df)

In [ ]:
marking.check('p1-first-five', first_5=first_5)

In [ ]:
marking.check("p1-shape", data_shape=data_shape)

In [ ]:
marking.check('p1-statistics', data_statis=data_statis)

In [ ]:
marking.check('p1-missing', data_missing_summary=data_missing_summary)

## Part 2. Bar charts, a line chart, and a heatmap (40 marks)

Save the named axes for each plot. Each chart has its own 10-mark check.

### A. Total Sales by Channel
Plot vertical bars with total Sales descending. Break ties alphabetically by Channel.
Use axes `ax_channel_sales` and labels `Channel` and `Sales`.

In [ ]:
### BEGIN SOLUTION
sales_by_channel = df.groupby("Channel")["Sales"].sum().sort_values(
    ascending=False, kind="stable"
)
fig, ax_channel_sales = plt.subplots(figsize=(8, 4))
ax_channel_sales.bar(sales_by_channel.index, sales_by_channel.values)
ax_channel_sales.set(title="Total Sales by Channel", xlabel="Channel", ylabel="Sales")
ax_channel_sales.tick_params(axis="x", labelrotation=45)
fig.tight_layout()
plt.show()
### END SOLUTION

In [ ]:
marking.check('p2-channel-sales', ax_channel_sales=ax_channel_sales)

### B. Spend vs. Sales by Region
Plot side-by-side vertical bars of total Spend and Sales for each region, including `Unknown`.
Use axes `ax_region_spend_sales`, labels `Region` and `Total value`, and legend entries `Spend` and `Sales`.

In [ ]:
### BEGIN SOLUTION
region_totals = df.groupby("Region")[["Spend", "Sales"]].sum()
x = np.arange(len(region_totals))
width = 0.38
fig, ax_region_spend_sales = plt.subplots(figsize=(8, 4))
ax_region_spend_sales.bar(x - width / 2, region_totals["Spend"], width, label="Spend")
ax_region_spend_sales.bar(x + width / 2, region_totals["Sales"], width, label="Sales")
ax_region_spend_sales.set_xticks(x, region_totals.index)
ax_region_spend_sales.set(title="Spend vs. Sales by Region", xlabel="Region", ylabel="Total value")
ax_region_spend_sales.legend()
fig.tight_layout()
plt.show()
### END SOLUTION

In [ ]:
marking.check('p2-region-bars', ax_region_spend_sales=ax_region_spend_sales)

### C. Monthly Sales trend
Plot one line showing total Sales by month, in chronological order. Use dates at the start of each month on the x-axis.
Use axes `ax_monthly_sales` and labels `Month` and `Sales`.

In [ ]:
### BEGIN SOLUTION
months = pd.to_datetime(df["Month"]).dt.to_period("M").dt.to_timestamp()
monthly_sales = df.groupby(months)["Sales"].sum().sort_index()
fig, ax_monthly_sales = plt.subplots(figsize=(8, 4))
ax_monthly_sales.plot(monthly_sales.index, monthly_sales.values, marker="o")
ax_monthly_sales.set(title="Monthly Sales Trend", xlabel="Month", ylabel="Sales")
fig.autofmt_xdate()
fig.tight_layout()
plt.show()
### END SOLUTION

In [ ]:
marking.check('p2-monthly-line', ax_monthly_sales=ax_monthly_sales)

### D. Correlation heatmap
Use Pearson correlation for `Spend`, `Leads`, `Sales`, `Impressions`, and `Clicks`, in that order.
Use `imshow` on axes `ax_correlation` and label both sets of ticks with the variable names.

In [ ]:
### BEGIN SOLUTION
measures = ["Spend", "Leads", "Sales", "Impressions", "Clicks"]
correlation = df[measures].corr()
fig, ax_correlation = plt.subplots(figsize=(7, 6))
image = ax_correlation.imshow(correlation, cmap="RdBu_r", vmin=-1, vmax=1)
ax_correlation.set_xticks(range(len(measures)), measures, rotation=45, ha="right")
ax_correlation.set_yticks(range(len(measures)), measures)
ax_correlation.set_title("Correlation Heatmap")
fig.colorbar(image, ax=ax_correlation)
fig.tight_layout()
plt.show()
### END SOLUTION

In [ ]:
marking.check('p2-heatmap', ax_correlation=ax_correlation)

## Part 3. Channel CTR and CPL (10 marks)
Sum Clicks, Impressions, Spend, and Leads within each channel first. Calculate CTR as total Clicks divided by total Impressions, and CPL as total Spend divided by total Leads. Use `NaN` where a denominator is zero.
- Save `by_chan` with columns `Channel`, `CTR`, `CPL`, in that order (6 marks).
- Save the highest CTR channel as `max_ctr_channel` (2 marks) and the lowest CPL channel as `min_cpl_channel` (2 marks). For either answer, any tied leader is valid.

In [ ]:
### BEGIN SOLUTION
totals = df.groupby("Channel", as_index=False)[["Clicks", "Impressions", "Spend", "Leads"]].sum()
totals["CTR"] = totals["Clicks"] / totals["Impressions"].replace(0, np.nan)
totals["CPL"] = totals["Spend"] / totals["Leads"].replace(0, np.nan)
by_chan = totals[["Channel", "CTR", "CPL"]]
max_ctr_channel = by_chan.loc[by_chan["CTR"].idxmax(), "Channel"]
min_cpl_channel = by_chan.loc[by_chan["CPL"].idxmin(), "Channel"]
display(by_chan)
print("Highest CTR:", max_ctr_channel)
print("Lowest CPL:", min_cpl_channel)
### END SOLUTION

In [ ]:
marking.check('p3-channel-rates', by_chan=by_chan)

In [ ]:
marking.check('p3-best-ctr', max_ctr_channel=max_ctr_channel)

In [ ]:
marking.check('p3-lowest-cpl', min_cpl_channel=min_cpl_channel)

## Part 4. Sales-to-Spend by Region (10 marks)
Sum Sales and Spend within each region. Divide total Sales by total Spend, using `NaN` when total Spend is zero.
- Save `by_region` with `Region` as the index and columns `Sales`, `Spend`, `Sales_to_Spend`, in that order (4 marks).
- Save the leading region as `max_ratio_region` (6 marks). Any tied leader is valid.

In [ ]:
### BEGIN SOLUTION
by_region = df.groupby("Region")[["Sales", "Spend"]].sum()
by_region["Sales_to_Spend"] = by_region["Sales"] / by_region["Spend"].replace(0, np.nan)
max_ratio_region = by_region["Sales_to_Spend"].idxmax()
display(by_region)
print("Highest Sales-to-Spend ratio:", max_ratio_region)
### END SOLUTION

In [ ]:
marking.check('p4-region-ratios', by_region=by_region)

In [ ]:
marking.check('p4-best-region', max_ratio_region=max_ratio_region)

## Part 5. Top five channels by total Sales (10 marks)
Save `top5_channel` as a Series of total Sales with Channel as its index. Sort descending, break ties alphabetically, then keep the first five (5 marks).
Plot the selected values as vertical bars on `ax_top5`, using the same order and labels `Channel` and `Sales` (5 marks).

In [ ]:
### BEGIN SOLUTION
top5_channel = df.groupby("Channel")["Sales"].sum().sort_values(
    ascending=False, kind="stable"
).head(5)
fig, ax_top5 = plt.subplots(figsize=(7, 4))
ax_top5.bar(top5_channel.index, top5_channel.values)
ax_top5.set(title="Top 5 Channels by Sales", xlabel="Channel", ylabel="Sales")
ax_top5.tick_params(axis="x", labelrotation=45)
fig.tight_layout()
plt.show()
### END SOLUTION

In [ ]:
marking.check('p5-top-five', top5_channel=top5_channel)

In [ ]:
marking.check('p5-bar', ax_top5=ax_top5)

## Part 6. Impressions vs. Clicks (20 marks)
Plot all rows with finite values for both Impressions and Clicks as a scatter plot on `ax_scatter`. Use `Impressions` on x and `Clicks` on y, with those exact axis labels (10 marks).

In the written answer, describe the direction, strength, and spread or outliers (10 marks).

-- Hint: OLS trend line is optional, but it could help you to identify the trend. (OLS has no marks)

In [ ]:
### BEGIN SOLUTION
fig, ax_scatter = plt.subplots(figsize=(6, 4))
pairs = df[["Impressions", "Clicks"]]
valid = np.isfinite(pairs).all(axis=1)
ax_scatter.scatter(pairs.loc[valid, "Impressions"], pairs.loc[valid, "Clicks"], alpha=0.7)
ax_scatter.set(title="Impressions vs. Clicks", xlabel="Impressions", ylabel="Clicks")
fig.tight_layout()
plt.show()
### END SOLUTION

In [ ]:
marking.check('p6-scatter', ax_scatter=ax_scatter)

Clicks generally rise as impressions increase. The Pearson correlation is about 0.64, consistent with a moderately strong positive relationship. Points at similar impression counts have a noticeable spread of click counts, so impressions alone do not determine clicks. I would inspect the points with unusually high or low clicks for their impression levels before treating them as outliers.

## Part 7. Actionable recommendation (10 manual marks)
Propose one specific action for the marketing team, supported by a finding from your analysis. Explain what the team should do and how the finding supports it.

Pilot a modest increase in Display impressions and compare CTR with a comparable campaign that keeps its budget unchanged. Display has the highest aggregate CTR in the supplied data, about 3.13%, so it is a reasonable channel to test for engagement. Track CPL and sales as well to see whether additional clicks translate into useful outcomes before expanding the campaign.

In [ ]:
marking.summarise_marks()